## Final loss across many models

This notebook loads multiple saved models and collects `training_params['ll'][-1]` into a table.

- Edit `model_dirs` (or use the glob-based discovery).
- Run all cells.
- Output is saved as `../figures/model_final_losses.csv` (relative to `generate_figures/`).


In [ ]:
import os,sys
import socket
from pathlib import Path
import numpy as np
import pandas as pd
import torch

np.random.seed(0)
torch.manual_seed(0)

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
from IPython.display import display

from vi_rnn.saving import load_model, CPU_Unpickler
from vi_rnn.generate import generate
from vi_rnn.data_utils import make_all_trials
from vi_rnn.utils import get_orth_proj_latents

import pyvista as pv
pv.set_jupyter_backend('static') # Options: 'static', 'trame', or 'panel'

import matplotlib as mpl
import seaborn as sns
%matplotlib inline



In [ ]:
from fig_utils.basii import (
    compute_var_explained,
    make_train_test_split,
    project_position_at_decode,
    format_data_for_marg_pca_trials,
    compute_orthogonal_subspace,
)
from fig_utils.plots import (
    plot_basis_2d_subspaces,
    plot_basis_3d_trajectories,
    plot_time_latents,
    plot_variance_explained_by_macaque,
    plot_variance_explained_components,
)

In [ ]:
hostname = socket.gethostname()
print("hostname:", hostname)

if hostname == "MatthijsDesktop":
    out_dir = Path("/home/matthijs/swm_rnn/final_models/macaque")
    path = "/home/matthijs/swm_rnn/data/"

else:
    out_dir = Path("/Users/matthijs/swm_rnn_cl/final_models/macaque")
    path = str(Path.cwd().parent / "data") + "/"


model_dirs = [
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_28_T_05_34_01",
    "SWM_low_rank_one_to_one_dim_z_64_date_2026_05_01_T_22_03_56",
    "SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_18_03_12",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_17_02_13",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_30_36",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_26_38",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_27_25",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_30_22",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_20_35",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_13_05",
]

In [ ]:
# --- controls ---
# compute variance explained window

# --- task ---
n_pos = 3  # number of positions
n_stim = 6  # number of stimuli

# --- data ---
k = 20  # number of duplicate runs

# --- windows ---
t_start_w1 = 20  # start of the sequence position 1 window
t_start_w2 = 33  # start of the sequence position 2 window
t_start_w3 = 45  # start of the sequence position 3 window
t_start_t = 45  # 20  # start of the temporal basis window
var_t1 = 45
var_t2 = -1

# --- style ---
plot_means = True
generate_plots = True

# --- run ---
split_seperate_conds = False  # use held out conditions for computing basii
train_size = 0.5  # fraction of data to use for training
noise_scale = 1  # noise scale for generating data
basis = "pca"  # use PCA basis
orth_basis = True  # orthogonalize basis between seperate marginal PCAs
n_pcs_time = 2  # amount of temporal basis directions
marg_order = ["t", "s1", "s2", "s3"]  # compute order for orthogonal basis
run = False

In [ ]:
time_windows = [[t_start_t, -1], [t_start_w1, -1], [t_start_w2, -1], [t_start_w3, -1]]
t_min = min(t_start_t, t_start_w1, t_start_w2, t_start_w3)

In [ ]:
# get task_params of one of the models
task_params_file = str(out_dir) + "/" + model_dirs[0] + "_task_params.pkl"
with open(task_params_file, "rb") as f:
    task_params = CPU_Unpickler(f).load()

u, _, labels_training, delay_ends = make_all_trials(
    task_params,
    dur=3.55,
    n_stim=6,
    n_pos=n_pos,
    cue_dur=-1,
    bin_size=0.05,
    interval_dur="mean",
    delay_dur="mean",
)  # ,ramp_amp=.75,ramp_dur=-1)

# np.random.shuffle(labels_training) # sanity check, that with random shuffling, the results are unintelligable

u = torch.tensor(u, dtype=torch.float32)
labels_training = np.int_(labels_training)

In [ ]:
plt.imshow(u[0])

In [ ]:
# Z: (repeats, conditions, time, latent_dim)
# labels_training: (conditions, n_factors)

In [ ]:
if run:
    rows = []

    for model_dir in model_dirs:
        model_dir = out_dir / Path(model_dir)
        name = model_dir.name

        vae, training_params, task_params = load_model(
            str(model_dir), load_encoder=True, backward_compat=False
        )

        # Generate data, and project on orthonormal basis from weights

        Zo, v, _, rates = generate(
            vae, u=u, x=None, k=k, noise_scale=noise_scale, initial_state="prior_mean"
        )  # training_params["k"])
        # Zo shape: (120, 64, 71, 4)
        projection_matrix = get_orth_proj_latents(vae)
        projection_matrix = projection_matrix.cpu().numpy()
        Z = np.einsum("ij,bjtk->kbit", projection_matrix, Zo)  # (k,120,64,71)
        print("Max and min of Zo:")
        print(torch.max(Zo), torch.min(Zo))

        # Make train test split
        (
            Z_train,
            Z_test,
            labels_train,
            labels_test,
            Z_train_mean,
            Z_test_mean,
            labels_train_mean,
            labels_test_mean,
            held_in_conditions,
            held_out_conditions,
        ) = make_train_test_split(Z, labels_training, train_size, split_seperate_conds)
        X_full, unique_levels, weights = format_data_for_marg_pca_trials(
            Z_train_mean, labels_train_mean, 1
        )

        weights[..., :t_min] = 0
        R = X_full.mean(
            axis=0
        )  # (64,6,6,6,71) dim_z, stimuli_pos1, stimuli_pos2, stimuli_pos3, time

        mean = np.sum(
            R * weights[None, ...], axis=(1, 2, 3, 4), keepdims=True
        ) / np.sum(weights[None, ...], axis=(1, 2, 3, 4), keepdims=True)
        X_full -= mean
        R -= mean
        mean = mean.squeeze()
        (
            transform,
            inv_transform,
            W_enc,
            W_full,
            global_mean,
            scaling,
        ) = compute_orthogonal_subspace(
            R,
            mask=np.bool_(weights),
            marg_order=marg_order,
            n_components=[n_pcs_time, 2, 2, 2],
            time_windows=time_windows,
            center=True,
            center_marginals=True,
            orthogonalize=orth_basis,
            basis=basis,
        )
        Z_red = transform(X_full)
        Z_red = Z_red[0]

        # check if global_mean is close to zero
        if np.linalg.norm(global_mean) > 1e-4:
            print("global_mean is not close to zero")
        else:
            print("global_mean is close to zero")

        var_explained = compute_var_explained(
            Z_test, W_enc, var_t1, var_t2, n_pcs_time, n_pos
        )

        # store the full transform
        A_comb_np = W_full.T @ np.diag(1 / scaling) @ projection_matrix
        b_comb_np = W_full.T @ np.diag(1 / scaling) @ mean

        # --------------------------------
        # plotting
        # --------------------------------
        if generate_plots:
            cmap = mpl.colors.ListedColormap(sns.color_palette("husl", n_colors=6))
            t_decode = -1
            if plot_means:
                Z_plot, labels_plot = Z_test_mean, labels_test_mean
            else:
                Z_plot, labels_plot = Z_test, labels_test

            z_by_pos = [
                project_position_at_decode(transform, Z_plot, t_decode, i, n_pcs_time)
                for i in range(n_pos)
            ]
            c_by_pos = [labels_plot[:, i] for i in range(n_pos)]
            plot_basis_2d_subspaces(
                z_by_pos, c_by_pos, cmap=cmap, n_stim=n_stim, n_pos=n_pos
            )

            Z_T = transform(Z_plot)
            plot_time_latents(Z_T, n_pcs_time, n_trials=Z_T.shape[0])

            for pos_ind in range(n_pos):
                plot_basis_3d_trajectories(
                    Z_T,
                    labels_plot,
                    pos_ind=pos_ind,
                    cmap=cmap,
                    n_pcs_time=n_pcs_time,
                )

        rows.append(
            {
                "name": name,
                "path": str(model_dir),
                "macaque": task_params["sessions"][0][5:10],
                "var_explained": var_explained,
                "A_comb_np": A_comb_np,
                "b_comb_np": b_comb_np,
                "held_in_conditions": held_in_conditions,
                "held_out_conditions": held_out_conditions,
            }
        )

In [ ]:
if run:
    df = pd.DataFrame(rows)
    # store the df
    df.to_pickle("../data/processed/df_basii.pkl")
else:
    df = pd.read_pickle("../data/processed/df_basii.pkl")

In [ ]:
plot = plot_variance_explained_by_macaque(
    df,
    n_pcs_time=n_pcs_time,
    save_path="../paper_figures/variance_explained_by_components.pdf",
    box_w=1.6,
    box_h=0.6,
)

In [ ]:
var_all = np.stack(df["var_explained"].values)
var_explained = var_all.mean(axis=0)
var_err = var_all.std(axis=0, ddof=1)

plot_variance_explained_components(
    var_explained,
    var_err,
    n_pcs_time=n_pcs_time,
    box_w=1.6,
    box_h=0.6,
    dpi=300,
    save_path="../paper_figures/variance_explained_by_components.pdf",
    show=True,
)

In [ ]:
np.mean(var_all.sum(axis=1)) * 100

In [ ]:
np.std(var_all.sum(axis=1)) * 100